# IndicDLP Live Layout Studio: Hybrid GPU + Gemini OCR Auto-Annotator

Automated batch pipeline combining YOLOv10x layout detection on local GPU (RTX 2050 CUDA) with Gemini Flash Lite for high-accuracy historical Bengali OCR, exported into pre-annotated Label Studio tasks.

## 1. Environment Setup & Dependencies

In [ ]:
import os
import sys
import json
import time
import uuid
import torch
from PIL import Image
from google import genai


## 2. Path Configurations & Model Initialization

In [ ]:
REPO_DIR = r'C:\Users\shiny\Documents\IndicDLP-Live-Layout-Studio'
INPUT_DIR = r'C:\Users\shiny\Downloads\Newspaper_Scans\Extracted\test_10'
OUTPUT_JSON = r'C:\Users\shiny\Desktop\label_studio_preannotated_test10.json'

sys.path.append(REPO_DIR)
from model_inference import load_model, predict_once

API_KEY = os.environ.get('GEMINI_API_KEY', '')
client = genai.Client(api_key=API_KEY)
OCR_MODEL = 'gemini-2.5-flash'

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if 'cuda' in device else 'CPU'
print(f'Device: {device} ({gpu_name})')

model = load_model('yolov10x', device=device)


## 3. Label Studio Class Taxonomy Mapping

In [ ]:
CLASS_MAP = {
    'headline': 'Headline',
    'title': 'Headline',
    'flag': 'Headline',
    'header': 'Headline',
    'sub-title': 'Subtitle',
    'sub-headline': 'Subtitle',
    'heading': 'Subtitle',
    'section-title': 'Subtitle',
    'sub-section-title': 'Subtitle',
    'paragraph': 'BodyText',
    'ordered-list': 'BodyText',
    'unordered-list': 'BodyText',
    'table': 'BodyText',
    'table-of-contents': 'BodyText',
    'index': 'BodyText',
    'sidebar': 'BodyText',
    'advertisement': 'BodyText',
    'caption': 'Caption',
    'table-caption': 'Caption',
    'figure-caption': 'Caption',
    'figure': 'Image'
}


## 4. Vintage Bengali Transcription Engine

In [ ]:
def transcribe_crop_with_gemini(crop_img, label):
    if label == 'Image':
        return ''
    
    prompt = 'Transcribe the Bengali text in this cropped historical newspaper snippet verbatim. Do not add commentary or notes. Output only the exact transcribed text:'
    for attempt in range(2):
        try:
            response = client.models.generate_content(
                model=OCR_MODEL,
                contents=[crop_img, prompt]
            )
            text = response.text.strip() if response and response.text else ''
            return text
        except Exception:
            time.sleep(1.5)
    return 


## 5. Batch Auto-Annotation & Label Studio JSON Packaging

In [ ]:
def process_scans():
    image_files = sorted([f for f in os.listdir(INPUT_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'Found {len(image_files)} scans in: {INPUT_DIR}')
    
    tasks = []

    for idx, fname in enumerate(image_files):
        fpath = os.path.join(INPUT_DIR, fname)
        print(f'[{idx+1}/{len(image_files)}] Processing: {fname}')
        
        t0 = time.time()
        orig_img = Image.open(fpath).convert('RGB')
        w_orig, h_orig = orig_img.size
        
        raw_dets = predict_once('yolov10x', orig_img, conf=0.22, iou=0.45, imgsz=1024, device=device)
        print(f'  YOLO found {len(raw_dets)} regions')

        mapped_regions = []
        for d in raw_dets:
            lbl = d['label'].lower()
            mapped_lbl = CLASS_MAP.get(lbl, 'BodyText')
            mapped_regions.append({
                'box': d['box'],
                'label': mapped_lbl,
                'conf': d['conf']
            })

        def region_priority(r):
            x1, y1, x2, y2 = r['box']
            area = (x2 - x1) * (y2 - y1)
            priority_score = 0
            if r['label'] == 'Headline': priority_score = 3
            elif r['label'] == 'Subtitle': priority_score = 2
            elif r['label'] == 'Caption': priority_score = 1
            return (priority_score, area)

        mapped_regions = sorted(mapped_regions, key=region_priority, reverse=True)

        results = []
        ocr_budget = min(10, len(mapped_regions))

        for r_idx, r in enumerate(mapped_regions):
            x1, y1, x2, y2 = r['box']
            bw = x2 - x1
            bh = y2 - y1
            
            x_pct = (x1 / w_orig) * 100.0
            y_pct = (y1 / h_orig) * 100.0
            w_pct = (bw / w_orig) * 100.0
            h_pct = (bh / h_orig) * 100.0

            region_id = str(uuid.uuid4())[:8]

            transcription = ''
            if r_idx < ocr_budget and r['label'] != 'Image':
                crop = orig_img.crop((int(x1), int(y1), int(x2), int(y2)))
                transcription = transcribe_crop_with_gemini(crop, r['label'])
                time.sleep(0.4)

            bbox_result = {
                'id': region_id,
                'type': 'rectanglelabels',
                'value': {
                    'x': x_pct,
                    'y': y_pct,
                    'width': w_pct,
                    'height': h_pct,
                    'rotation': 0,
                    'rectanglelabels': [r['label']]
                },
                'to_name': 'image',
                'from_name': 'label',
                'image_rotation': 0,
                'original_width': w_orig,
                'original_height': h_orig
            }
            results.append(bbox_result)

            if transcription:
                text_result = {
                    'id': region_id,
                    'type': 'textarea',
                    'value': {
                        'x': x_pct,
                        'y': y_pct,
                        'width': w_pct,
                        'height': h_pct,
                        'rotation': 0,
                        'text': [transcription]
                    },
                    'to_name': 'image',
                    'from_name': 'transcription'
                }
                results.append(text_result)

        task = {
            'data': {
                'image': f'/data/local-files/?d={fname}'
            },
            'predictions': [
                {
                    'model_version': 'gemini-ocr-1',
                    'result': results
                }
            ]
        }
        tasks.append(task)
        print(f'  Done in {time.time() - t0:.2f}s')

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(tasks, f, ensure_ascii=False, indent=2)

    print(f'Saved {len(tasks)} tasks to: {OUTPUT_JSON}')

process_scans()
